## Tutorial 3 - Preparing for production

To deal with any steric clashes that could have been present in the input system or introduced in the coarse-graining stage or in bilayer building, we need to energy-minimise the system. For coarse-grained systems, this will normally suffice to be able to seed a production run, but care should be taken to disregard the start of the production simulation as the system equilibrates.

The most widely used simulation engine for Martini simulations is GROMACS, which we will use here. There is now an implementation of [Martini in OpenMM](https://www.cell.com/biophysj/fulltext/S0006-3495(23)00237-0?_returnURL=https%3A%2F%2Flinkinghub.elsevier.com%2Fretrieve%2Fpii%2FS0006349523002370%3Fshowall%3Dtrue) which can also be used.   

The `grompp` step in GROMACS is used to prepare a production file, which is an output in the form of a `.tpr` file. The inputs are:
- `-c` a coordinate file (`.pdb` or `.gro`)
- `-p` the topology file, showing what is contained within the system and information about the forcefield/parameter files (`.top` file)
- `-f` an `.mdp` file, which contains the settings for this simulation run
- `-maxwarn` which suppresses warnings. **This should not be used unless you know what you are doing**, but used below to dismiss warnings about atom name changes (which happened with new lipid parameters) and another warning that we can disregard for now

We can have a look at what is contained within this energy minimisation file:

In [ ]:
!head mdps/em.mdp

- `integrator` defines what type of simulation is performed; in this case it uses the steepest decents algorithm to perform energy minimisation  
- `nsteps` defines the maximum number of steps performed  
- `emtol` is the cutoff where we define the system as energy minimised, a target energy (in kJ mol<sup>-1</sup>)  
- `emstep` is the largest displacement allowed per step performed (in nm)  

We can now use this file and the files we created in the previous step to generate a `.tpr` file

In [ ]:
%%bash 

cp ../tutorial_2/system.gro ../tutorial_2/topol.top .

cp ../tutorial_2/*.itp  itps/

gmx grompp -f mdps/em.mdp -c system.gro -p topol.top -o em.tpr -maxwarn 2

We can now run our energy minimisation. We do this with the `gmx mdrun` command, which takes the following inputs:

- `-deffnm` sets the default file names and uses this to find the .tpr file
- `-v` to be verbose and print all the steps it is taking (and when it might finish)
- `-ntmpi` number of thread-MPI ranks to start

In [ ]:
%%bash

gmx mdrun -deffnm em -v -ntmpi 1

We now have an energy minimised system! We can have a look at this in VMD and hopefully you can see the difference:

Do this **out of the notebook** in your own terminal:

```$vmd em.gro```

We can now use this to generate our production simulation. The `.mdp` file listing the settings is a lot longer and more complicated, we can see this below. 

In [ ]:
%%bash

head -n 44 mdps/5us-martini.mdp

We can highlight the most important settings here

- `integrator` this time is md, which will use an algorithm for integrating Newton's equation of motion (in this case using a leap-frog algorithm).
- `dt` which is the time step used by the integrator. For atomistic simulations, this is often 2 fs, but with Martini this can be extended to 20 fs.
- `nsteps` which, with the md integrator, is the number of steps that will happen. As we are now using a time-based integrator, we can calculate that 250000000 x 2 fs = 5 $\mu$s.
- `nstxout-compressed` is how many steps between saving coordinates into the `.xtc` format. As it stands, it is saved every nanosecond.
- `Pcoupltype` is the type of isotropy used for the pressure coupling. For soluble simulations, this will usually be set to isotropic, where each dimension (x,y and z) would be treated uniquely. When there is a membrane present, the x and y dimensions are intrinsically coupled, so we need to use the semiisotropic pressure coupling setting.
- `Pcouple` is the pressure coupling type to use, in this case the C-rescale algorithm.
- `tcoupl` is the temperature coupling type, in this case the v-rescale algorithm.
- `tc-groups` specifies groups to separate temperature baths. Separating into similarly mobile parts can help prevent excessive energies accumulating in one component.

To be able to determine the groups used in `tc-groups`, we need to make an index file. We can do that using a GROMACS command:

In [ ]:
%%bash

gmx make_ndx -f em.gro -o sys.ndx << EOF
rPOPC|rPOPE|rPOPI
name 18 LIPID
rW|rION
name 19 SOL_ION

q
EOF

Since the Protein is already listed in the index file, we do not need to specify this further. We need to group the lipids (**r**esidue POPE |-or etc) together and give this the appropriate name; same with the lipids and ions.

We are now ready to simulate our system! Depending on what you are studying, the number of repeats and length of simulation could differ. For investigating something such as lipid-protein interaction, I would usually start with 5 x 5 $\mu$s simulations, especially for a system of this size. We can set up one simulation below:

In [ ]:
%%bash

gmx grompp -f mdps/5us-martini.mdp -c em.gro -p topol.top -n sys.ndx -o md.tpr

Because of time constraints, here is a simulation I prepared earlier so we can move on to analysis of coarse-grained simulations